# Задание №7: Реализация дерева решений, случайного леса и градиентного бустинга

В данном ноутбуке представлена реализация следующих алгоритмов машинного обучения с нуля на Python с использованием библиотеки `numpy` и соблюдением принципов ООП:
* **DecisionTree** — дерево решений (алгоритм CART) для задач классификации и регрессии.
* **BaseEnsemble** — базовый абстрактный класс для ансамблевых моделей.
* **RandomForest** — случайный лес (бэггинг + метод случайных подпространств).
* **GradientBoosting** — градиентный бустинг над деревьями решений.

## 1. Импорт библиотек

In [ ]:
from abc import ABC, abstractmethod
import numpy as np

## 2. Класс узла и дерева решений (`Node`, `DecisionTree`)

In [ ]:
class Node:
    """Класс узла дерева решений."""

    def __init__(
        self,
        feature=None,
        threshold=None,
        left=None,
        right=None,
        *,
        value=None,
    ):
        self.feature = feature      # Индекс признака для разбиения
        self.threshold = threshold  # Пороговое значение
        self.left = left            # Левое поддерево
        self.right = right          # Правое поддерево
        self.value = value          # Значение в листе (если узел листовой)

    def is_leaf(self) -> bool:
        return self.value is not None


class DecisionTree:
    """Реализация дерева решений (CART) для классификации и регрессии."""

    def __init__(
        self,
        task: str = "classification",
        max_depth: int = 10,
        min_samples_split: int = 2,
        n_features: int = None,
    ):
        if task not in ("classification", "regression"):
            raise ValueError(
                "Задача должна быть 'classification' или 'regression'."
            )

        self.task = task
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.root = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        self.n_features = (
            X.shape[1]
            if not self.n_features
            else min(X.shape[1], self.n_features)
        )
        self.root = self._grow_tree(X, y)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _grow_tree(self, X: np.ndarray, y: np.ndarray, depth: int = 0) -> Node:
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        # Критерии остановки рекурсии
        if (
            depth >= self.max_depth
            or n_labels == 1
            or n_samples < self.min_samples_split
        ):
            leaf_value = self._calculate_leaf_value(y)
            return Node(value=leaf_value)

        feat_idxs = np.random.choice(n_feats, self.n_features, replace=False)

        best_feature, best_threshold = self._best_split(X, y, feat_idxs)

        if best_feature is None:
            return Node(value=self._calculate_leaf_value(y))

        left_idxs, right_idxs = self._split(X[:, best_feature], best_threshold)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return Node(value=self._calculate_leaf_value(y))

        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return Node(best_feature, best_threshold, left, right)

    def _best_split(self, X: np.ndarray, y: np.ndarray, feat_idxs: np.ndarray):
        best_gain = -1.0
        split_idx, split_threshold = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)

            for thr in thresholds:
                gain = self._information_gain(y, X_column, thr)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_threshold = thr

        return split_idx, split_threshold

    def _information_gain(
        self, y: np.ndarray, X_column: np.ndarray, threshold: float
    ) -> float:
        parent_loss = self._impurity(y)
        left_idxs, right_idxs = self._split(X_column, threshold)

        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0.0

        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = self._impurity(y[left_idxs]), self._impurity(y[right_idxs])
        child_loss = (n_l / n) * e_l + (n_r / n) * e_r

        return parent_loss - child_loss

    def _impurity(self, y: np.ndarray) -> float:
        if self.task == "classification":
            # Критерий Джини
            _, counts = np.unique(y, return_counts=True)
            probabilities = counts / len(y)
            return 1.0 - np.sum(probabilities**2)
        else:
            # Среднеквадратичная ошибка (MSE) для регрессии
            return np.var(y) if len(y) > 0 else 0.0

    def _split(self, X_column: np.ndarray, split_thresh: float):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def _calculate_leaf_value(self, y: np.ndarray):
        if self.task == "classification":
            vals, counts = np.unique(y, return_counts=True)
            return vals[np.argmax(counts)]
        else:
            return np.mean(y)

    def _traverse_tree(self, x: np.ndarray, node: Node):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

## 3. Базовый класс ансамблей (`BaseEnsemble`)

In [ ]:
class BaseEnsemble(ABC):
    """Абстрактный базовый класс для ансамблевых моделей."""

    def __init__(
        self,
        n_estimators: int = 10,
        max_depth: int = 5,
        min_samples_split: int = 2,
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.trees = []

    @abstractmethod
    def fit(self, X: np.ndarray, y: np.ndarray):
        pass

    @abstractmethod
    def predict(self, X: np.ndarray) -> np.ndarray:
        pass

## 4. Случайный лес (`RandomForest`)

In [ ]:
class RandomForest(BaseEnsemble):
    """Случайный лес для классификации и регрессии."""

    def __init__(
        self,
        task: str = "classification",
        n_estimators: int = 10,
        max_depth: int = 10,
        min_samples_split: int = 2,
        max_features: int = None,
    ):
        super().__init__(n_estimators, max_depth, min_samples_split)
        self.task = task
        self.max_features = max_features

    def fit(self, X: np.ndarray, y: np.ndarray):
        self.trees = []
        n_samples, n_feats = X.shape

        if self.max_features is None:
            self.max_features = (
                int(np.sqrt(n_feats))
                if self.task == "classification"
                else max(1, n_feats // 3)
            )

        for _ in range(self.n_estimators):
            tree = DecisionTree(
                task=self.task,
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                n_features=self.max_features,
            )
            idxs = np.random.choice(n_samples, n_samples, replace=True)
            tree.fit(X[idxs], y[idxs])
            self.trees.append(tree)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        tree_preds = np.swapaxes(tree_preds, 0, 1)

        if self.task == "classification":
            predictions = []
            for preds in tree_preds:
                vals, counts = np.unique(preds, return_counts=True)
                predictions.append(vals[np.argmax(counts)])
            return np.array(predictions)
        else:
            return np.mean(tree_preds, axis=1)

## 5. Градиентный бустинг (`GradientBoosting`)

In [ ]:
class GradientBoosting(BaseEnsemble):
    """Градиентный бустинг над деревьями решений (бинарная классификация и регрессия)."""

    def __init__(
        self,
        task: str = "regression",
        n_estimators: int = 50,
        learning_rate: float = 0.1,
        max_depth: int = 3,
        min_samples_split: int = 2,
    ):
        super().__init__(n_estimators, max_depth, min_samples_split)
        if task not in ("classification", "regression"):
            raise ValueError(
                "Задача должна быть 'classification' или 'regression'."
            )
        self.task = task
        self.learning_rate = learning_rate
        self.init_prediction = None

    @staticmethod
    def _sigmoid(z: np.ndarray) -> np.ndarray:
        return 1.0 / (1.0 + np.exp(-np.clip(z, -250, 250)))

    def fit(self, X: np.ndarray, y: np.ndarray):
        self.trees = []

        if self.task == "regression":
            self.init_prediction = np.mean(y)
            current_pred = np.full(y.shape, self.init_prediction, dtype=float)
        else:
            pos_prob = np.clip(np.mean(y), 1e-15, 1 - 1e-15)
            self.init_prediction = np.log(pos_prob / (1 - pos_prob))
            current_pred = np.full(y.shape, self.init_prediction, dtype=float)

        for _ in range(self.n_estimators):
            if self.task == "regression":
                residuals = y - current_pred
            else:
                probs = self._sigmoid(current_pred)
                residuals = y - probs

            tree = DecisionTree(
                task="regression",
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
            )
            tree.fit(X, residuals)

            update = tree.predict(X)
            current_pred += self.learning_rate * update
            self.trees.append(tree)

        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        pred = np.full(X.shape[0], self.init_prediction, dtype=float)
        for tree in self.trees:
            pred += self.learning_rate * tree.predict(X)

        if self.task == "regression":
            return pred
        else:
            probs = self._sigmoid(pred)
            return (probs >= 0.5).astype(int)

## 6. Тестирование моделей на синтетических данных

In [ ]:
np.random.seed(42)

# 1. Тест на задаче РЕГРЕССИИ
X_reg = np.random.uniform(-5, 5, size=(200, 2))
y_reg = np.sin(X_reg[:, 0]) * 3 + X_reg[:, 1] ** 2 + np.random.normal(0, 0.5, 200)

X_train_r, X_test_r = X_reg[:150], X_reg[150:]
y_train_r, y_test_r = y_reg[:150], y_reg[150:]

print("=== РЕГРЕССИЯ (Метрика: MSE) ===")
for name, model in [
    ("Decision Tree", DecisionTree(task="regression", max_depth=6)),
    ("Random Forest", RandomForest(task="regression", n_estimators=20, max_depth=6)),
    ("Gradient Boosting", GradientBoosting(task="regression", n_estimators=50, learning_rate=0.1, max_depth=3)),
]:
    model.fit(X_train_r, y_train_r)
    preds = model.predict(X_test_r)
    mse = np.mean((y_test_r - preds) ** 2)
    print(f"{name:18s} | MSE: {mse:.4f}")

# 2. Тест на задаче БИНАРНОЙ КЛАССИФИКАЦИИ
X_clf = np.random.randn(200, 4)
y_clf = ((X_clf[:, 0] + X_clf[:, 1] * 1.5 - X_clf[:, 2]) > 0).astype(int)

X_train_c, X_test_c = X_clf[:150], X_clf[150:]
y_train_c, y_test_c = y_clf[:150], y_clf[150:]

print("\n=== КЛАССИФИКАЦИЯ (Метрика: Accuracy) ===")
for name, model in [
    ("Decision Tree", DecisionTree(task="classification", max_depth=5)),
    ("Random Forest", RandomForest(task="classification", n_estimators=20, max_depth=5)),
    ("Gradient Boosting", GradientBoosting(task="classification", n_estimators=50, learning_rate=0.1, max_depth=3)),
]:
    model.fit(X_train_c, y_train_c)
    preds = model.predict(X_test_c)
    acc = np.mean(y_test_c == preds)
    print(f"{name:18s} | Accuracy: {acc:.4f}")